# 03 · Técnicas de conteo y probabilidad (sección 5.10, Tema 3)

Escenarios con universos explícitos, todos sobre la muestra elegible (n = 5.230 registros; la unidad es el
registro de línea, no el cliente único):

| Técnica | Escenario |
|---|---|
| Principio multiplicativo | Perfiles posibles al combinar Banca × Billetera × Pagos (y la restricción de contacto) |
| Permutaciones | Órdenes de prioridad de las 3 campañas digitales; calendario de 3 meses con 5 campañas candidatas |
| Combinaciones | Grupos de campañas sin orden; grupos de registros para un piloto |
| Probabilidad y frecuencia relativa | Probabilidad de elegir al azar un registro con cada preferencia; probabilidades condicionales |
| Selección sin reemplazo (hipergeométrica) | Probabilidad de que una selección aleatoria de m registros contenga x con banca digital |

In [1]:
import math
from itertools import permutations, product
import numpy as np
import pandas as pd
from scipy import stats
import config

el = config.leer("elegibles")
N = len(el)
B = el.BAN_BANCA == True          # noqa: E712
W = el.BAN_BILLETERA == True      # noqa: E712
P = el.BAN_PAGOS_ONLINE == True   # noqa: E712
S = el.BAN_SUPRESION_I == True    # noqa: E712  (True = restricción de mensajes activa)
validacion = []


def validar(indicador, manual, computacional, tolerancia, observacion):
    dif = abs(manual - computacional)
    validacion.append({"indicador": indicador, "manual": manual, "computacional": computacional, "diferencia": dif,
                       "tolerancia": tolerancia, "cumple": bool(dif <= tolerancia), "observacion": observacion})

## 5.10.1 Principio multiplicativo
Cada indicador tiene 2 estados (activo / inactivo): Total = 2 · 2 · 2 = 8 perfiles teóricos.
Luego se cuenta cuántos se observan realmente.

In [2]:
perfiles = []
for b_, w_, p_ in product([True, False], repeat=3):
    cnt = int(((B == b_) & (W == w_) & (P == p_)).sum())
    perfiles.append({"banca": b_, "billetera": w_, "pagos": p_, "registros": cnt, "pct": cnt / N,
                     "etiqueta": " + ".join(n_ for n_, v in [("Banca", b_), ("Billetera", w_), ("Pagos", p_)] if v) or "Ninguna"})
perfiles = pd.DataFrame(perfiles).sort_values("registros", ascending=False)
total_teorico = 2 * 2 * 2
observados = int((perfiles.registros > 0).sum())
con_supresion = total_teorico * 2
validar("Perfiles teóricos (2·2·2)", float(total_teorico), float(len(list(product([0, 1], repeat=3)))), 0,
        "producto manual vs enumeración con itertools.product")
print(f"Perfiles teóricos: {total_teorico} · observados con al menos un registro: {observados}")
perfiles

Perfiles teóricos: 8 · observados con al menos un registro: 8


,banca,billetera,pagos,registros,pct,etiqueta
7,False,False,False,4862,0.929637,Ninguna
3,True,False,False,162,0.030975,Banca
0,True,True,True,92,0.017591,Banca + Billetera + Pagos
2,True,False,True,83,0.015870,Banca + Pagos
4,False,True,True,12,0.002294,Billetera + Pagos
6,False,False,True,11,0.002103,Pagos
1,True,True,False,4,0.000765,Banca + Billetera
5,False,True,False,4,0.000765,Billetera


## 5.10.2 Permutaciones sin repetición: P(n, m) = n! / (n − m)!
**Escenario A:** ordenar las 3 campañas digitales (Banca, Billetera, Pagos) de primera a tercera: P(3,3).
**Escenario B:** 5 campañas candidatas (las 3 digitales + Fidelización por antigüedad + Ajuste de cargo fijo) y
3 meses de calendario; cada mes una campaña distinta: P(5,3).

In [3]:
campanas = ["Banca digital", "Billetera", "Pagos en línea"]
ordenes_A = [list(o) for o in permutations(campanas)]
P33 = math.factorial(3) // math.factorial(0)
P53 = math.factorial(5) // math.factorial(5 - 3)
validar("P(3,3)", float(P33), float(len(ordenes_A)), 0, "n!/(n−m)! vs enumeración itertools.permutations")
validar("P(5,3)", float(P53), float(math.perm(5, 3)), 0, "n!/(n−m)! vs math.perm")
# orden sugerido por los datos: de mayor a menor proporción activa
orden_datos = sorted(campanas, key=lambda c: -{"Banca digital": B, "Billetera": W, "Pagos en línea": P}[c].mean())
print(f"P(3,3) = {P33} órdenes · P(5,3) = {P53} calendarios · orden por proporción observada: {orden_datos}")

P(3,3) = 6 órdenes · P(5,3) = 60 calendarios · orden por proporción observada: ['Banca digital', 'Pagos en línea', 'Billetera']


## 5.10.3 Combinaciones sin repetición: C(n, m) = n! / [m!(n − m)!]
**Escenario C:** elegir 3 de las 5 campañas sin importar el orden: C(5,3).
**Escenario D:** grupos distintos de 10 registros para un piloto, tomados de los registros contactables con banca
digital activa: C(K, 10).

In [4]:
C53 = math.factorial(5) // (math.factorial(3) * math.factorial(2))
validar("C(5,3)", float(C53), float(math.comb(5, 3)), 0, "fórmula factorial vs math.comb")
validar("P(5,3) = C(5,3)·3!", float(P53), float(C53 * math.factorial(3)), 0, "relación permutación–combinación")
K_piloto = int((B & ~S).sum())
C_piloto = math.comb(K_piloto, 10)
print(f"C(5,3) = {C53} · registros contactables con banca: {K_piloto} · C({K_piloto},10) = {C_piloto:.3e}")

C(5,3) = 10 · registros contactables con banca: 194 · C(194,10) = 1.644e+16


## 5.10.4 Probabilidad y frecuencia relativa
Al escoger **un registro al azar** de los 5.230 (resultados equiprobables), P(E) = casos favorables / posibles,
que coincide con la frecuencia relativa p̂ = f / n.

In [5]:
def prob(mask):
    return int(mask.sum()) / N


probs = {
    "P(Banca)": prob(B), "P(Billetera)": prob(W), "P(Pagos)": prob(P), "P(Restricción)": prob(S),
    "P(Banca ∩ Pagos)": prob(B & P), "P(Banca ∪ Pagos)": prob(B | P),
    "P(al menos una financiera)": prob(B | W | P), "P(ninguna financiera)": prob(~(B | W | P)),
    "P(Pagos | Banca)": int((B & P).sum()) / int(B.sum()),
    "P(Banca | contactable)": int((B & ~S).sum()) / int((~S).sum()),
    "P(Banca | restringido)": int((B & S).sum()) / int(S.sum()),
}
validar("P(Banca ∪ Pagos) por suma de probabilidades", probs["P(Banca)"] + probs["P(Pagos)"] - probs["P(Banca ∩ Pagos)"],
        probs["P(Banca ∪ Pagos)"], 1e-12, "P(A)+P(B)−P(A∩B) vs conteo directo")
validar("p̂ Banca", int(B.sum()) / N, float(el.BAN_BANCA.value_counts(normalize=True)[True]), 1e-12,
        "f/n manual vs value_counts(normalize=True)")
pd.Series(probs).round(4)

P(Banca)                      0.0652
P(Billetera)                  0.0214
P(Pagos)                      0.0379
P(Restricción)                0.8763
P(Banca ∩ Pagos)              0.0335
P(Banca ∪ Pagos)              0.0696
P(al menos una financiera)    0.0704
P(ninguna financiera)         0.9296
P(Pagos | Banca)              0.5132
P(Banca | contactable)        0.2998
P(Banca | restringido)        0.0321
dtype: float64

## 5.10.5 Selección sin reemplazo: modelo hipergeométrico
P(X = x) = C(K, x)·C(N − K, m − x) / C(N, m)

**Escenario E:** se seleccionan al azar m = 20 registros de los N = 5.230 elegibles; K = registros con banca activa.
**Escenario F:** la misma selección, pero solo entre los registros **contactables** (sin restricción).

In [6]:
def hiper_manual(Nn, K, m, x):
    return math.comb(K, x) * math.comb(Nn - K, m - x) / math.comb(Nn, m)


m_sel = 20
escenarios = {}
for nombre, universo in [("E", np.ones(N, bool)), ("F", (~S).to_numpy())]:
    Nn, K = int(universo.sum()), int((B.to_numpy() & universo).sum())
    dist = [hiper_manual(Nn, K, m_sel, x) for x in range(m_sel + 1)]
    rv = stats.hypergeom(Nn, K, m_sel)
    validar(f"Hipergeométrica {nombre}: P(X=1)", dist[1], float(rv.pmf(1)), 1e-12, "combinaciones manuales vs scipy.stats.hypergeom")
    escenarios[nombre] = {"N": Nn, "K": K, "m": m_sel, "dist": dist, "P0": dist[0], "P1": dist[1],
                          "P_al_menos_1": 1 - dist[0], "P_al_menos_3": float(sum(dist[3:])),
                          "P_al_menos_5": float(sum(dist[5:])), "esperado": m_sel * K / Nn,
                          "media_scipy": float(rv.mean()), "moda": int(np.argmax(dist))}
pd.DataFrame({k_: {c: v for c, v in e.items() if c != "dist"} for k_, e in escenarios.items()}).T

,N,K,m,P0,P1,P_al_menos_1,P_al_menos_3,P_al_menos_5,esperado,media_scipy,moda
E,5230.0,341.0,20.0,0.258978,0.362676,0.741022,0.137852,0.007851,1.304015,1.304015,1.0
F,647.0,194.0,20.0,0.000705,0.006298,0.999295,0.966450,0.766010,5.996909,5.996909,6.0


In [7]:
# Conteos para la calculadora del dashboard (universo × indicador → N y K)
universos = {"Todos los elegibles": np.ones(N, bool), "Contactables": (~S).to_numpy(), "Con restricción": S.to_numpy()}
indicadores = {"Banca digital": B.to_numpy(), "Billetera": W.to_numpy(), "Pagos en línea": P.to_numpy(),
               "Alguna financiera": (B | W | P).to_numpy()}
conteos = [{"universo": u, "indicador": i, "N": int(mu.sum()), "K": int((mi & mu).sum())}
           for u, mu in universos.items() for i, mi in indicadores.items()]
config.guardar_json({
    "N": N, "perfiles": perfiles.to_dict("records"), "total_teorico": total_teorico, "observados": observados,
    "con_supresion": con_supresion, "campanas": campanas, "ordenes_A": ordenes_A, "P33": P33, "P53": P53, "C53": C53,
    "orden_datos": orden_datos, "K_piloto": K_piloto, "C_piloto_log10": math.log10(C_piloto),
    "C_muestra_log10": math.log10(math.comb(N, 20)),
    "probs": probs, "escenarios": escenarios, "conteos": conteos, "validacion": validacion,
}, "r03_probabilidad")
pd.DataFrame(validacion)

,indicador,manual,computacional,diferencia,tolerancia,cumple,observacion
0,Perfiles teóricos (2·2·2),8.000000,8.000000,0.000000e+00,0.000000e+00,True,producto manual vs enumeración con itertools.p...
1,"P(3,3)",6.000000,6.000000,0.000000e+00,0.000000e+00,True,n!/(n−m)! vs enumeración itertools.permutations
2,"P(5,3)",60.000000,60.000000,0.000000e+00,0.000000e+00,True,n!/(n−m)! vs math.perm
3,"C(5,3)",10.000000,10.000000,0.000000e+00,0.000000e+00,True,fórmula factorial vs math.comb
4,"P(5,3) = C(5,3)·3!",60.000000,60.000000,0.000000e+00,0.000000e+00,True,relación permutación–combinación
5,P(Banca ∪ Pagos) por suma de probabilidades,0.069598,0.069598,1.387779e-17,1.000000e-12,True,P(A)+P(B)−P(A∩B) vs conteo directo
6,p̂ Banca,0.065201,0.065201,0.000000e+00,1.000000e-12,True,f/n manual vs value_counts(normalize=True)
7,Hipergeométrica E: P(X=1),0.362676,0.362676,1.110223e-16,1.000000e-12,True,combinaciones manuales vs scipy.stats.hypergeom
8,Hipergeométrica F: P(X=1),0.006298,0.006298,8.673617e-19,1.000000e-12,True,combinaciones manuales vs scipy.stats.hypergeom
